In [1]:
#environment setup
import pandas as pd
import numpy as np
import joblib
import optuna
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

train_df = pd.read_csv('../data/processed/train_final.csv')
test_df = pd.read_csv('../data/processed/test_final.csv')
bc_lambda = joblib.load('../models/bc_lambda.pkl')

BEST_WINDOW = 12  # from Phase 5 findings

print("Train shape:", train_df.shape, "| Test shape:", test_df.shape)


def safe_inv_boxcox(y_pred, lmbda, max_reasonable=1500):
    """Safely invert a Box-Cox transform, clipping to avoid domain errors and unrealistic values."""
    if lmbda != 0:
        valid = np.clip(1 + lmbda * y_pred, a_min=1e-6, a_max=None)
        result = valid ** (1 / lmbda)
    else:
        result = np.exp(y_pred)
    return np.clip(result, a_min=0, a_max=max_reasonable)


def evaluate_predictions(y_true, y_pred, model_name):
    """Compute MAE, RMSE, MAPE, and R² for a set of predictions."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-6, None))) * 100
    r2 = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}


def create_sequences(X, y, window):
    """Convert tabular data into overlapping sequences of length `window` for RNN input."""
    Xs, ys = [], []
    for i in range(len(X) - window):
        Xs.append(X[i:i+window])
        ys.append(y[i+window])
    return np.array(Xs), np.array(ys)

C:\Users\Welcome\OneDrive - NSBM\Desktop\3rd_year\vectorium labs\appliance-energy-prediction-dl-\energy_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train shape: (15771, 48) | Test shape: (3930, 48)


In [2]:
#Build feature/target sets (tabular, for XGBoost)

feature_cols = [c for c in train_df.columns if c not in ['date', 'Appliances', 'Appliances_bc']]

X_train = train_df[feature_cols]
y_train = train_df['Appliances_bc']
X_test = test_df[feature_cols]
y_test_actual = test_df['Appliances']

print("Features:", len(feature_cols))

Features: 45


In [3]:
#Load prior baseline/DL results (for before/after comparison later)

baseline_results = pd.read_csv('../data/processed/baseline_results.csv')
dl_results_df = pd.read_csv('../data/processed/dl_results.csv')

print(baseline_results)
print(dl_results_df)

               Model        MAE        RMSE       MAPE        R2
0              Naive  26.246819   65.747028  21.898901  0.449627
1  Linear Regression  39.004480  131.875555  26.403592 -1.214287
2      Random Forest  21.467393   52.669068  16.502867  0.646803
3            XGBoost  19.286535   48.798904  15.277149  0.696803
     Model        MAE       RMSE       MAPE        R2
0     LSTM  28.281775  68.642078  22.916919  0.388753
1      GRU  32.139880  82.197495  25.676301  0.123497
2  CNN-GRU  31.489904  73.430912  25.237913  0.300490


In [4]:
#Optuna: tune XGBoost

def objective_xgb(trial):
    """Optuna objective for XGBoost hyperparameter search using time-series CV."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': 42
    }

    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(mean_absolute_error(y_val, pred))

    return np.mean(scores)

study_xgb = optuna.create_study(direction='minimize', study_name='xgb_tuning')
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

print("Best XGBoost params:", study_xgb.best_params)
print("Best CV MAE (Box-Cox space):", study_xgb.best_value)

[I 2026-07-19 20:27:28,605] A new study created in memory with name: xgb_tuning
Best trial: 0. Best value: 0.0223064:   3%|▎         | 1/30 [00:21<10:29, 21.71s/it]

[I 2026-07-19 20:27:50,305] Trial 0 finished with value: 0.022306437098892438 and parameters: {'n_estimators': 365, 'max_depth': 10, 'learning_rate': 0.10738379804638151, 'subsample': 0.6177186431132018, 'colsample_bytree': 0.6537985594470426, 'min_child_weight': 3}. Best is trial 0 with value: 0.022306437098892438.


Best trial: 1. Best value: 0.0202844:   7%|▋         | 2/30 [00:24<04:52, 10.45s/it]

[I 2026-07-19 20:27:52,876] Trial 1 finished with value: 0.02028440278859803 and parameters: {'n_estimators': 153, 'max_depth': 5, 'learning_rate': 0.024000975801239506, 'subsample': 0.9952616195816354, 'colsample_bytree': 0.6795652122684726, 'min_child_weight': 7}. Best is trial 1 with value: 0.02028440278859803.


Best trial: 2. Best value: 0.0189287:  10%|█         | 3/30 [00:39<05:44, 12.76s/it]

...

[I 2026-07-19 20:32:54,577] Trial 27 finished with value: 0.019120071183726547 and parameters: {'n_estimators': 412, 'max_depth': 9, 'learning_rate': 0.048625413024488605, 'subsample': 0.867945740970222, 'colsample_bytree': 0.9845836311474019, 'min_child_weight': 10}. Best is trial 26 with value: 0.018854701599407674.


Best trial: 26. Best value: 0.0188547:  97%|█████████▋| 29/30 [05:46<00:16, 16.63s/it]

[I 2026-07-19 20:33:15,411] Trial 28 finished with value: 0.020369920663724315 and parameters: {'n_estimators': 447, 'max_depth': 10, 'learning_rate': 0.08098792772601682, 'subsample': 0.6869741372310493, 'colsample_bytree': 0.9567246036301775, 'min_child_weight': 8}. Best is trial 26 with value: 0.018854701599407674.


Best trial: 26. Best value: 0.0188547: 100%|██████████| 30/30 [06:08<00:00, 12.29s/it]

[I 2026-07-19 20:33:37,161] Trial 29 finished with value: 0.020109023373942776 and parameters: {'n_estimators': 499, 'max_depth': 10, 'learning_rate': 0.1112884897431442, 'subsample': 0.9772763829161624, 'colsample_bytree': 0.8501695421476644, 'min_child_weight': 9}. Best is trial 26 with value: 0.018854701599407674.
Best XGBoost params: {'n_estimators': 415, 'max_depth': 9, 'learning_rate': 0.027254568594193486, 'subsample': 0.9722348355713633, 'colsample_bytree': 0.9697175164202116, 'min_child_weight': 9}
Best CV MAE (Box-Cox space): 0.018854701599407674


In [5]:
#Retrain XGBoost with best params, evaluate on real test set

best_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
best_xgb.fit(X_train, y_train)
pred_actual = safe_inv_boxcox(best_xgb.predict(X_test), bc_lambda)

tuned_xgb_metrics = evaluate_predictions(y_test_actual, pred_actual, 'XGBoost_Tuned')
print("Tuned XGBoost:", tuned_xgb_metrics)
print("Original XGBoost:", baseline_results[baseline_results['Model']=='XGBoost'].to_dict('records')[0])

joblib.dump(best_xgb, '../models/xgb_tuned.pkl')

Tuned XGBoost: {'Model': 'XGBoost_Tuned', 'MAE': 17.914650167493416, 'RMSE': np.float64(44.86144363123109), 'MAPE': np.float64(14.611852494575903), 'R2': 0.7437571706446189}
Original XGBoost: {'Model': 'XGBoost', 'MAE': 19.2865354070943, 'RMSE': 48.7989044873842, 'MAPE': 15.277148960197916, 'R2': 0.6968026676410302}


['../models/xgb_tuned.pkl']

In [6]:
#LSTM Optimization


In [7]:
#Prepare sequence data for LSTM tuning

X_train_raw = train_df[feature_cols].values
y_train_raw = train_df['Appliances_bc'].values
X_test_raw = test_df[feature_cols].values
y_test_bc_raw = test_df['Appliances_bc'].values
y_test_actual_full = test_df['Appliances'].values

Xs, ys = create_sequences(X_train_raw, y_train_raw, BEST_WINDOW)
Xte, yte_bc = create_sequences(X_test_raw, y_test_bc_raw, BEST_WINDOW)
yte_actual = y_test_actual_full[BEST_WINDOW:]

print("Sequence shapes:", Xs.shape, Xte.shape)

Sequence shapes: (15759, 12, 45) (3918, 12, 45)


In [ ]:
#Optuna: tune LSTM

def objective_lstm(trial):
    """Optuna objective for LSTM hyperparameter search."""
    units = trial.suggest_int('units', 32, 128)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(Xs.shape[1], Xs.shape[2])),
        LSTM(units//2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='huber')

    es = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
    history = model.fit(Xs, ys, validation_split=0.15, epochs=30, batch_size=batch_size,
                         callbacks=[es], verbose=0)
    return min(history.history['val_loss'])

study_lstm = optuna.create_study(direction='minimize', study_name='lstm_tuning')
study_lstm.optimize(objective_lstm, n_trials=20, show_progress_bar=True)

print("Best LSTM params:", study_lstm.best_params)

[I 2026-07-19 20:52:37,487] A new study created in memory with name: lstm_tuning
  0%|          | 0/20 [00:00<?, ?it/s]

C:\Users\Welcome\OneDrive - NSBM\Desktop\3rd_year\vectorium labs\appliance-energy-prediction-dl-\energy_env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
